# 02 · Aprendizaje supervisado: predecir el nivel de tráfico
**TráficoIA · Proyecto Final, versión 3 · Semana 2 (sábado 21 de noviembre)**

Equipo N.º ____ · Integrantes: ______________________

**Pregunta:** dadas la hora, el día de la semana, la temperatura, la lluvia y si es festivo, ¿el tráfico será **bajo, medio o alto**?

**Método (y por qué):**
1. Los niveles se definen con terciles del volumen, **calculados solo con entrenamiento**, para que la prueba no influya en la definición de las clases.
2. Separación 80 % entrenamiento / 20 % prueba.
3. Se comparan cuatro algoritmos (árbol, Random Forest, KNN y regresión logística) más un **modelo base** que siempre responde la clase más frecuente. Si un modelo no supera al base, no aprendió nada.
4. Cada modelo ajusta sus hiperparámetros con **validación cruzada de 5 particiones dentro de entrenamiento**. El mejor modelo se elige por F1 macro de validación cruzada, **nunca mirando la prueba**.
5. La prueba se usa una sola vez, para reportar el resultado final.

> **Antes de ejecutar:** haber descargado los datos (`python src/descargar_datos.py --tabular`). Este cuaderno tarda unos minutos por la búsqueda de hiperparámetros.

In [ ]:
import sys, warnings
from pathlib import Path

# Permite importar src/ aunque el cuaderno se ejecute desde notebooks/
RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, supervisado

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
config.FIGURAS.mkdir(parents=True, exist_ok=True)

def guardar(nombre):
    plt.tight_layout()
    plt.savefig(config.FIGURAS / f"{nombre}.png", dpi=130, bbox_inches="tight")

print("Raíz del proyecto:", RAIZ)

## 1. Datos, variable objetivo y separación

In [ ]:
df = supervisado.cargar_datos_modelado()
print(f"{len(df):,} horas | {df['date_time'].min()}  ->  {df['date_time'].max()}")

part = supervisado.dividir(df)
print(f"Entrenamiento: {len(part.X_train):,} filas | Prueba: {len(part.X_test):,} filas")
print("Variables de entrada:", part.features)
print("Cortes de nivel (calculados SOLO con entrenamiento): "
      "bajo <= {:.0f} < medio <= {:.0f} < alto  (vehículos por hora)".format(*part.cortes))

pd.DataFrame({"entrenamiento": part.y_train.value_counts(normalize=True),
              "prueba": part.y_test.value_counts(normalize=True)}).reindex(config.NIVELES).round(3)

*Interpretación:* las tres clases quedan casi equilibradas (≈ 1/3 cada una) porque se definieron con terciles, así que **exactitud y F1 macro serán parecidas** y un modelo que adivine siempre la misma clase obtendrá ≈ 0.33 de exactitud. Ese es el piso que hay que superar. Escribe aquí cualquier diferencia que veas entre entrenamiento y prueba.

## 2. Comparar algoritmos con validación cruzada
Para cada modelo se prueba una rejilla pequeña de hiperparámetros con 5 particiones. La tabla muestra el mejor de cada uno:

* `f1_macro_cv`: rendimiento en datos que el modelo no vio durante cada ajuste (promedio de las 5 particiones) y su desviación.
* `brecha_sobreajuste`: F1 de entrenamiento menos F1 de validación. Una brecha grande indica que el modelo memoriza.

In [ ]:
resultados = supervisado.buscar_mejores(part)      # tarda 1 a 3 minutos
tabla = supervisado.tabla_cv(resultados)
tabla.round(3)

In [ ]:
sin_base = tabla[tabla["modelo"] != supervisado.BASE]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(tabla["modelo"], tabla["f1_macro_cv"], xerr=tabla["f1_macro_cv_desv"], color="#2a6f97", capsize=4)
ax.set(title="F1 macro en validación cruzada (5 particiones)", xlabel="F1 macro", xlim=(0, 1))
ax.invert_yaxis()
for i, v in enumerate(tabla["f1_macro_cv"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
guardar("sup_comparacion_cv")
plt.show()

*Interpretación:* compara cada modelo contra el base. ¿Cuál queda primero y por cuánto frente al segundo? ¿La diferencia es mayor que la desviación entre particiones (las barras de error)? Si no lo es, **no se puede afirmar** que uno sea mejor que otro. La regresión logística suele quedar muy atrás: la relación entre hora y tráfico no es lineal (dos picos al día), y un modelo lineal no la captura.

## 3. Resultado en el conjunto de prueba (se mira una sola vez)

In [ ]:
tabla_prueba, predicciones = supervisado.evaluar_en_prueba(resultados, part)
comparativa = (tabla.drop(columns="mejores_parametros")
                    .merge(tabla_prueba, on="modelo")
                    .set_index("modelo"))
comparativa[["exactitud_cv", "f1_macro_cv", "exactitud_prueba", "f1_macro_prueba", "brecha_sobreajuste"]].round(3)

*Interpretación:* los valores de prueba deben ser parecidos a los de validación cruzada. Si el rendimiento en prueba fuera mucho menor, habría sobreajuste o fuga de información; si es parecido, la validación cruzada fue una buena estimación.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

modelos_mostrar = [m for m in comparativa.index if m != supervisado.BASE]
fig, ejes = plt.subplots(1, len(modelos_mostrar), figsize=(4.2 * len(modelos_mostrar), 4))
for ax, m in zip(np.atleast_1d(ejes), modelos_mostrar):
    ConfusionMatrixDisplay.from_predictions(part.y_test, predicciones[m], labels=config.NIVELES,
                                            normalize="true", cmap="Blues", ax=ax, colorbar=False,
                                            values_format=".2f")
    ax.set_title(m, fontsize=10)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real" if ax is np.atleast_1d(ejes)[0] else "")
fig.suptitle("Matrices de confusión en prueba (proporción por fila)", y=1.02)
guardar("sup_matrices_confusion")
plt.show()

## 4. Elegir el mejor modelo
La elección se hace con la **validación cruzada** (entrenamiento), no con la prueba.

In [ ]:
candidatos = tabla[tabla["modelo"] != supervisado.BASE]
mejor = candidatos.loc[candidatos["f1_macro_cv"].idxmax(), "modelo"]
print("Mejor modelo por F1 macro de validación cruzada:", mejor)
print("Hiperparámetros:", resultados[mejor].best_params_)
print()
print(classification_report(part.y_test, predicciones[mejor], labels=config.NIVELES, digits=3))

*Interpretación:* si dos modelos quedaron empatados dentro de la desviación, ¿por qué elegiste este? Criterios válidos: menor brecha de sobreajuste, más sencillo de explicar, o más rápido para predecir en la app. Escríbelo aquí.

## 5. Sobreajuste
El árbol de decisión es el caso clásico: sin límite de profundidad memoriza el entrenamiento. La curva muestra cómo cambia el F1 al dejarlo crecer.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

kf = StratifiedKFold(5, shuffle=True, random_state=config.SEMILLA)
filas = []
for d in range(1, 26):
    arbol = Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("m", DecisionTreeClassifier(max_depth=d, random_state=config.SEMILLA))])
    f_cv = cross_val_score(arbol, part.X_train, part.y_train, cv=kf, scoring="f1_macro", n_jobs=-1).mean()
    f_tr = arbol.fit(part.X_train, part.y_train).score(part.X_train, part.y_train)
    filas.append({"profundidad": d, "entrenamiento": f_tr, "validacion_cruzada": f_cv})
curva = pd.DataFrame(filas)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(curva["profundidad"], curva["entrenamiento"], marker="o", ms=4, label="Entrenamiento (exactitud)")
ax.plot(curva["profundidad"], curva["validacion_cruzada"], marker="o", ms=4, label="Validación cruzada (F1 macro)")
ax.set(title="Árbol de decisión: profundidad y sobreajuste", xlabel="Profundidad máxima", ylabel="Métrica")
ax.legend()
guardar("sup_curva_profundidad")
plt.show()
print("Mejor profundidad por validación cruzada:", int(curva.loc[curva['validacion_cruzada'].idxmax(), 'profundidad']))

*Interpretación:* ¿a partir de qué profundidad el entrenamiento sigue subiendo pero la validación se estanca o baja? Ahí empieza el sobreajuste. Relaciona tu respuesta con la columna `brecha_sobreajuste` de la tabla.

## 6. Análisis de errores: ¿a qué horas y en qué condiciones se confunden los niveles?
Esta es la interpretación que pide la especificación. Se analiza el modelo elegido sobre el conjunto de prueba.

In [ ]:
err = part.X_test.copy()
err["hora"] = err["hora"].astype(int)
err["dia_semana"] = err["dia_semana"].astype(int)
err["real"] = part.y_test
err["predicho"] = predicciones[mejor]
err["error"] = err["real"] != err["predicho"]

print(f"Tasa de error global: {err['error'].mean():.1%}\n")
print("Confusiones (filas = real, columnas = predicho):")
print(pd.crosstab(err["real"], err["predicho"]).reindex(index=config.NIVELES, columns=config.NIVELES))

orden = {n: i for i, n in enumerate(config.NIVELES)}
distancia = (err["real"].map(orden) - err["predicho"].map(orden)).abs()
print(f"\nDe los errores, {(distancia[err['error']] == 1).mean():.1%} son entre niveles vecinos "
      f"(bajo-medio o medio-alto) y {(distancia[err['error']] == 2).mean():.1%} son bajo<->alto.")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))
por_hora = err.groupby("hora")["error"].mean()
ejes[0].bar(por_hora.index, por_hora.values * 100, color="#d62828")
ejes[0].set(title="Tasa de error por hora del día", xlabel="Hora", ylabel="Error (%)", xticks=range(0, 24, 2))

mapa = err.pivot_table(index="dia_semana", columns="hora", values="error", aggfunc="mean") * 100
mapa.index = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
sns.heatmap(mapa, cmap="Reds", ax=ejes[1], cbar_kws={"label": "Error (%)"})
ejes[1].set(title="Tasa de error por día y hora", xlabel="Hora", ylabel="")
guardar("sup_errores_hora")
plt.show()

print("Las 5 horas con más error:")
print((por_hora.sort_values(ascending=False).head(5) * 100).round(1).to_string())

In [ ]:
condiciones = pd.DataFrame({
    "Festivo": err.groupby(err["festivo_dia"].astype(bool))["error"].mean().rename({False: "No", True: "Sí"}),
    "Llueve (>0 mm)": err.groupby(err["rain_1h"] > 0)["error"].mean().rename({False: "No", True: "Sí"}),
    "Fin de semana": err.groupby(err["dia_semana"] >= 5)["error"].mean().rename({False: "No", True: "Sí"}),
}).T
(condiciones * 100).round(1).rename_axis("Condición").add_prefix("Error % si = ")

*Interpretación (obligatoria, escríbela con tus datos reales):*
* ¿En qué horas se concentran los errores? Normalmente son las **horas de transición** (cuando el tráfico sube de medio a alto por la mañana o baja por la tarde), donde el volumen está justo cerca de un corte.
* ¿Los errores son entre niveles vecinos o saltos de bajo a alto? Si casi todos son vecinos, el modelo falla por los bordes de las clases y no por entender mal el patrón.
* ¿Festivo o lluvia aumentan el error? Si sí, explica por qué (pocos ejemplos de festivos, efecto del clima).

## 7. Importancia de las variables

In [ ]:
from sklearn.inspection import permutation_importance

rf = resultados["Random Forest"].best_estimator_
imp_rf = pd.Series(rf.named_steps["m"].feature_importances_, index=part.features).sort_values()

muestra = part.X_test.sample(min(5000, len(part.X_test)), random_state=config.SEMILLA)
perm = permutation_importance(resultados[mejor].best_estimator_, muestra, part.y_test.loc[muestra.index],
                              scoring="f1_macro", n_repeats=5, random_state=config.SEMILLA, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=part.features).sort_values()

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
imp_rf.plot.barh(ax=ejes[0], color="#2a6f97")
ejes[0].set(title="Random Forest: importancia por impureza", xlabel="Importancia")
imp_perm.plot.barh(ax=ejes[1], color="#f4a261")
ejes[1].set(title=f"{mejor}: importancia por permutación (prueba)", xlabel="Caída del F1 macro al barajar la variable")
guardar("sup_importancia_variables")
plt.show()

*Interpretación:* ¿qué variable domina? Se espera que `hora` explique casi todo y que clima y temperatura aporten poco. La importancia por permutación es más fiable que la de impureza cuando hay variables con muchos valores distintos (como la temperatura).

## 8. Dos experimentos para validar el resultado
**a) Variables cíclicas.** La hora 23 y la hora 0 están pegadas, pero para el modelo son 23 unidades de distancia. Se prueba codificar hora y día con seno y coseno (afecta sobre todo a KNN).

**b) Validación temporal.** Una separación aleatoria deja horas vecinas (casi iguales) en entrenamiento y en prueba, lo que **favorece** al modelo. Entrenar con el pasado y probar con el futuro es más realista.

In [ ]:
df_ciclicas = supervisado.dividir(df, features=supervisado.FEATURES_CICLICAS)
df_temporal = supervisado.dividir(df, temporal=True)

filas = []
for m in modelos_mostrar:
    base_r = comparativa.loc[m, "f1_macro_prueba"]
    f_cic = supervisado.reentrenar_en(resultados, m, df_ciclicas)["f1_macro"]
    f_tmp = supervisado.reentrenar_en(resultados, m, df_temporal)["f1_macro"]
    filas.append({"modelo": m, "F1 aleatoria (base)": base_r, "F1 con variables cíclicas": f_cic,
                  "F1 validación temporal": f_tmp, "cambio temporal": f_tmp - base_r})
experimentos = pd.DataFrame(filas).set_index("modelo")
experimentos.round(3)

*Interpretación:* ¿las variables cíclicas mejoran a KNN? ¿Cuánto baja el F1 con validación temporal? Una caída grande significa que la separación aleatoria era optimista; una caída pequeña indica que el patrón horario es estable en el tiempo. En el informe final, reporta **ambos** números.

## 9. Guardar el modelo y probarlo

In [ ]:
metricas = {
    "f1_macro_cv": float(comparativa.loc[mejor, "f1_macro_cv"]),
    "exactitud_cv": float(comparativa.loc[mejor, "exactitud_cv"]),
    "f1_macro_prueba": float(comparativa.loc[mejor, "f1_macro_prueba"]),
    "exactitud_prueba": float(comparativa.loc[mejor, "exactitud_prueba"]),
    "f1_macro_temporal": float(experimentos.loc[mejor, "F1 validación temporal"]),
}
paquete = supervisado.crear_paquete(resultados, mejor, part, metricas)
ruta = supervisado.guardar_modelo(paquete)
print("Modelo guardado en:", ruta)

salida = config.RAIZ / "reports" / "tabla_comparativa_supervisado.csv"
comparativa.round(4).to_csv(salida)
print("Tabla comparativa guardada en:", salida)

In [ ]:
# Recargar y predecir: así lo usará la app de la semana 4
paquete = supervisado.cargar_modelo()
filas = []
for dia, nombre_dia in [(1, "Martes"), (5, "Sábado")]:
    for h in range(24):
        r = supervisado.predecir_nivel(paquete, hora=h, dia_semana=dia)
        filas.append({"dia": nombre_dia, "hora": h, "nivel": r["nivel"], "p_alto": r["probabilidades"]["alto"]})
prev = pd.DataFrame(filas)

fig, ax = plt.subplots(figsize=(9, 4))
for nombre_dia, g in prev.groupby("dia"):
    ax.plot(g["hora"], g["p_alto"], marker="o", ms=4, label=nombre_dia)
ax.set(title="Probabilidad de tráfico ALTO por hora (modelo guardado)", xlabel="Hora", ylabel="P(alto)",
       xticks=range(0, 24, 2))
ax.legend()
guardar("sup_prediccion_por_hora")
plt.show()

prev.pivot(index="hora", columns="dia", values="nivel").T

*Interpretación:* ¿las curvas tienen sentido (picos de mañana y tarde entre semana, y un fin de semana más bajo)? Si no lo tienen, hay un error que revisar antes de usar el modelo en la app.

## Conclusiones del módulo supervisado
*Completa con 4 a 6 hallazgos concretos y con números de TU ejecución:*

1. Mejor modelo y su F1 macro en validación cruzada y en prueba.
2. Qué tanto supera al modelo base y a la regresión logística, y por qué.
3. Dónde y por qué se confunden los niveles.
4. Qué variables importan y cuáles casi no.
5. Qué cambió con validación temporal y con variables cíclicas.
6. Limitaciones (datos de una sola autopista, clima simplificado, festivos) y qué mejorarías.